In [ ]:
import importlib

import evaluation
importlib.reload(evaluation)
from evaluation import load_and_evaluate_sequence

In [ ]:
from gauss_wasserstein import gauss_wasserstein_distance_

In [ ]:
from pathlib import Path
import json
import numpy as np
from stonesoup.metricgenerator.ospametric import GOSPAMetric
from stonesoup.types.state import GaussianState
from stonesoup.measures import Euclidean
from stonesoup.measures import Measure

In [ ]:
import pandas as pd

In [ ]:
class Wasserstein(Measure):
    def __call__(self, state1, state2):
        one = np.array(state1.state_vector).flatten(), np.array(state1.covar)
        
        two = np.array(state2.state_vector).flatten(), np.array(state2.covar)

        return gauss_wasserstein_distance_(one, two)

In [ ]:
def filter_velocity(prev, next, threshold):
    #pprint(prev)
    ptid = set([t["track_id"] for t in prev])
    ntid = set([t["track_id"] for t in next])
    data = []
    
    for tid in ptid.intersection(ntid):
        a = [t for t in prev if t["track_id"] == tid]
        b = [t for t in next if t["track_id"] == tid]
        #print(a, b)

        if len(a) != 1 or len(b) != 1:
            continue
        a = a[0]
        b = b[0]
        diff = np.linalg.norm(np.array(a["pos"]) - np.array(b["pos"]))
        #print(diff, threshold, diff < threshold)
        if diff < threshold:
            continue
        
        if "cov" in b:
            b["ext"] = b["cov"]

        data.append(b)
    return data

In [ ]:
def load_and_evaluate_sequence(path, threshold, p, c, measure="wasserstein"):
    # p = 1,c = 4 
    if measure == "euclidean":
        gospa = GOSPAMetric(p=p, c=c, measure=Euclidean())
    elif measure == "wasserstein":
        gospa = GOSPAMetric(p=p, c=c, measure=Wasserstein())
    else:
        raise AttributeError("please select a measure -> 'euclidean' or 'wasserstein'")
    
    data = {}
    for t in path.iterdir():
        name = t.name
        dd = {}
        for seq in sorted(list(t.iterdir())):
            if seq.name.startswith("."):
                continue
            with open(seq) as file:
                d = json.load(file)
                
                oldframe = None
                ddf = {}
                for frame in d["results"]:
                    if oldframe is None:
                        oldframe = frame
                        continue

                    gtf = filter_velocity(oldframe["gt"], frame["gt"], threshold)
                    trf = filter_velocity(oldframe["tr"], frame["tr"], threshold)
                    
                    gt = [GaussianState(d["pos"], d["ext"]) for d in gtf]
                    tr = [GaussianState(d["pos"], d["ext"]) for d in trf]

                    g = None
                    if len(gt) != 0 and len(tr) != 0:
                        g = gospa.compute_gospa_metric(tr, gt)
                    
                    ddf[frame["frame"]] = g
                    oldframe = frame
                    #print(frame["frame"])
                dd[seq.stem] = ddf
            data[name] = dd
    return data

In [ ]:
root = Path("./results")

latest = sorted(list(root.iterdir()), key=lambda x: x.stem)
last = latest[-1]
last

In [ ]:
last.stem

In [ ]:
%%time
m_pro_s = 0.0
data00 = load_and_evaluate_sequence(last, threshold=m_pro_s / 10, p=1, c=4)

In [ ]:
%%time
m_pro_s = 0.01
data001 = load_and_evaluate_sequence(last, threshold=m_pro_s / 10, p=1, c=4)

In [ ]:
%%time
m_pro_s = 0.1
data01 = load_and_evaluate_sequence(last, threshold=m_pro_s / 10, p=1, c=4)

In [ ]:
%%time
m_pro_s = 0.5
data05 = load_and_evaluate_sequence(last, threshold=m_pro_s / 10, p=1, c=4)

In [ ]:
%%time
m_pro_s = 1.0
data10 = load_and_evaluate_sequence(last, threshold=m_pro_s / 10, p=1, c=4)

In [ ]:
%%time
m_pro_s = 2.0
data20 = load_and_evaluate_sequence(last, threshold=m_pro_s / 10, p=1, c=4)

In [ ]:
def parse_gospa(gospa):
    return {"distance":gospa.value["distance"],
            "localisation":gospa.value["localisation"],
            "missed":gospa.value["missed"],
            "false":gospa.value["false"],
            "p":gospa.generator.p,
            "c":gospa.generator.c}
empty = {"distance":None,
            "localisation":None,
            "missed":None,
            "false":None,
            "p":None,
            "c":None}

In [ ]:
def asDataFrame(data):
    dt = []
    for method in data:
        for seq in data[method]:
            for frame in data[method][seq]:
                if data[method][seq][frame] is None:
                    dt.append({"method": method, "sequence": seq, "frame": frame,  **empty})
                    continue
                    
                gospa = data[method][seq][frame][0]
                dt.append({"method": method, "sequence": seq, "frame": frame,  **parse_gospa(gospa)})
    return pd.DataFrame(dt)

In [ ]:
df00 = asDataFrame(data00)
df00["veloth"] = 0.0
df001 = asDataFrame(data001)
df001["veloth"] = 0.01
df01 = asDataFrame(data01)
df01["veloth"] = 0.1
df05 = asDataFrame(data05)
df05["veloth"] = 0.5
df10 = asDataFrame(data10)
df10["veloth"] = 1.0
df20 = asDataFrame(data20)
df20["veloth"] = 2.0

df = pd.concat([df00, df001, df01, df05, df10, df20])

In [ ]:
tmp = Path("./temp")
tmp.mkdir(exist_ok=True)
tmp = tmp / f"{last.stem}_gw_v2.csv"
tmp

In [ ]:
df.to_csv(tmp, index=False)

In [ ]:
dfgw = pd.read_csv(tmp)
#df = pd.read_csv(f"./temp/{suffix}.csv")
df =dfgw

In [ ]:
df["intersection"] = pd.cut(df["sequence"], bins=[-1, 63, 125, 149], labels=[1, 2, 3])

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
plt.style.use("../../texed.mplstyle")
plt.rcParams['figure.figsize'] = (18, 14)
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=plt.cm.Dark2.colors)

In [ ]:
c = {"beta": "C0", "baseline":"C1"}
s = {0.0: "solid", 0.01: (0, (1, 1)), 0.1:"dotted", 0.5:(0, (1,5)), 1.0:"dashed", 2.0:"dotted"}

In [ ]:
corr = {"localisation": "$d_{localization}$",
        "distance": "$d_{GOSPA}$", 
        "missed":"$d_{missed}$", 
        "false":"$d_{false}$", 
        "beta": "Ours", "baseline": "Baseline"}

In [ ]:
xs = list(range(150))

fig, axs = plt.subplots(4, 4)
for i, (inter, tbl) in enumerate(df.groupby(["intersection"])):
    i = i+1
    for (velo,), dd in tbl.groupby(["veloth"]):#[["method", "sequence", "distance", "localisation", "missed", "false"]]:
        #print(dd.groupby("sequence").mean())
        if velo not in [0.0, 1.0, 2.0]:
            continue
        vals = ["distance", "localisation", "missed", "false"]
        for j, val in enumerate(vals):
            y_lim = df[val].max()
            
            for (m,), dt in dd.groupby(["method"]):
                #print(velo, val, m)
                
                axs[j, 0].set_ylabel(corr[val])
                v = dt.groupby("sequence")[[val]].mean().reset_index()
                xs = list(range(len(v[val])))
                
                axs[j, i].plot(xs, v[val].sort_values(ascending=False), color=c[m], linestyle=s[velo])# label=m
                axs[j, i].set_ylim(0, y_lim)
                #print(dt["sequence"], dt.groupby("sequence")[[val]])
                #.mean().plot.bar()


for (velo,), dd in df.groupby(["veloth"]):#[["method", "sequence", "distance", "localisation", "missed", "false"]]:
    #print(dd.groupby("sequence").mean())
    if velo not in [0.0, 1.0, 2.0]:
        continue
    vals = ["distance", "localisation", "missed", "false"]
    for j, val in enumerate(vals):
        y_lim = df[val].max()
        
        for (m,), dt in dd.groupby(["method"]):
            #print(velo, val, m)
            
            axs[j, 0].set_ylabel(corr[val])
            v = dt.groupby("sequence")[[val]].mean().reset_index()
            xs = list(range(len(v[val])))
            
            axs[j, 0].plot(xs, v[val].sort_values(ascending=False), color=c[m], linestyle=s[velo])# label=m
            axs[j, 0].set_ylim(0, y_lim)
            #print(dt["sequence"], dt.groupby("sequence")[[val]])
            #.mean().plot.bar()


axs[0, 0].set_title("total")
axs[0, 1].set_title("intersection 1")
axs[0, 2].set_title("intersection 2")
axs[0, 3].set_title("intersection 3")

plt.style.use("../../texed.mplstyle")
plt.savefig("results_overview.svg", bbox_inches='tight')
plt.show()

mh = [Line2D([0], [0], color=col, label=corr[m]) for m,col in c.items()]
legend1 = plt.legend(handles=mh, title="Methods", loc="upper left", ncols=2)#, bbox_to_anchor=(1.89 + 0.1, 4.68))
plt.gca().axis('off')
plt.tight_layout()
plt.savefig("leg1.pdf", bbox_inches="tight")
plt.gca().add_artist(legend1)

plt.show()

ph = [Line2D([0], [0], linestyle=ls, label=th, color="black") for th,ls in s.items() if th in [0.0, 1.0, 2.0]]
plt.legend(handles=ph, title="velocity threshold [m/s]", loc="upper left", ncols=6, columnspacing=0.8)#, bbox_to_anchor=(2 + 0.1, 3.87))
plt.gca().axis('off')
plt.tight_layout()
plt.savefig("leg2.pdf", bbox_inches="tight")
plt.show()
